# Diaformer inspired model with gdfs feature selection

gdfs selects which feature to look for next, and diaformer does the diagnosis

## setup

In [1]:
import os
import torch
import pandas as pd
import pickle
from torch import nn
from torch.optim import Adam
from torch.nn.functional import cross_entropy
from torch.nn.functional import mse_loss
import numpy as np
import json


In [2]:
root = os.path.join(r"C:\Users\Hzaab\SDP-Sequential-Diagnosis")
data = os.path.join(r"C:\Users\Hzaab\SDP-Sequential-Diagnosis\Casande-RL\Baselines\diaformer\data\ddxplus_10k")
metadata = os.path.join(root, r"afabench\Afabench_Source\data\release_evidences.json")
output = os.path.join(root, r"utilities\Model Development\outputs")
vocab = os.path.join(root, r"utilities\Model Development\vocab.txt")

device = torch.device("cuda")

budget=10

## Preprocessing

list all binary features

In [3]:
metadata_df = pd.read_json(metadata)

B_features = []
for ID,  definition in metadata_df.items():
    if definition["data_type"] == "B":
        B_features.append(ID)


create an index-name list for future reference (diaformer needs the name, gdfs needs numbers)

In [4]:
Bfeature_index = {}
for index, evidence_id in enumerate(B_features):
    Bfeature_index[evidence_id] = index

repeat for the diagnoses

In [5]:
with open(os.path.join(data, "vocab.txt"), encoding="utf-8") as f:
    text = f.read()

sections = text.split("\n\n")
diseases = sections[1].splitlines()

label_index = {}
for index, name in enumerate(diseases):
    label_index[name] = index


get the patients list

In [6]:
patients = pd.read_pickle(os.path.join(data, "goal_set.p"))
test_patients = pd.read_pickle(os.path.join(data, "test_data.pkl"))

patients is a list of dictionaries where each patient has: diseas, goal: explicit, implicit, option (answer)

make dataset skips non binary evidences and converts binary ones to 1/0, qns that arent answered by that patients will be hidden, initial evidence is included

In [7]:
def make_dataset(records):
    ans = []
    label = []
    starts = []

    for p in records:
        exp = p["goal"]["explicit_inform_slots"] 
        imp = p["goal"]["implicit_inform_slots"]

        given = exp.keys()
        init_name = list(given)[0]

        #remove any patient who has non binary init
        if init_name not in Bfeature_index:
            continue

        all_evidences = exp.copy()
        all_evidences.update(imp)

        row = []

        for feature in B_features:
            value = 0 

            if feature in all_evidences:
                value = all_evidences[feature]["option"]  #0=no, 1=yes

            row.append(value)

        ans.append(row)

        #convert the disease name into numeric label for gdfs
        label.append(label_index[p["disease_tag"]])

        starts.append(Bfeature_index[init_name])

    x = torch.tensor(ans) #all answers
    y = torch.tensor(label)       
    initial = torch.tensor(starts) 

    print("Keeping", len(ans), "of", len(records), "patients")

    return x, y, initial

run the function to generate the datsets

In [8]:
x_train, y_train, initial_train = make_dataset(patients["train"])
x_val, y_val, initial_val = make_dataset(patients["test"])
x_test, y_test, initial_test = make_dataset(test_patients["test"])

num_features = len(B_features)
num_diseases = len(diseases)
print("Binary questions:", num_features, "Diseases:", num_diseases)


Keeping 10000 of 10000 patients
Keeping 1000 of 1000 patients
Keeping 1000 of 1000 patients
Binary questions: 208 Diseases: 49


## Transformer

In [9]:
model = nn.ModuleDict()

#create embeddings for evidence, answer, and status
model["evidence"] = nn.Embedding(num_features + 1, 512) #+1 for summary token
model["answer"] = nn.Embedding(3, 512) #unknown, yes, no
model["status"] = nn.Embedding(3, 512) #unknown, init, given


layer = nn.TransformerEncoderLayer(
    d_model=512, #embedding size
    nhead=4,
    dim_feedforward=3072,
    batch_first=True
)

#create 5 transformer layers
model["transformer"] = nn.TransformerEncoder(layer, num_layers=5, enable_nested_tensor=False)
#output a score per disease
model["disease"] = nn.Linear(512, num_diseases)
#output a score per question
model["question"] = nn.Sequential(
    nn.Linear(512, 128),
    nn.ReLU(),
    nn.Linear(128, num_features)
)

model = model.to(device)

In [10]:
def forward(x, observed, initial):
    batch_size = len(x)
    rows = torch.arange(batch_size, device=device)

    #convert the answers into ids for the embedding layer
    #unknown = 0, no = 1, yes = 2
    answer_ids = x.long() + 1 #+1 so 0 is unknown, 1 is no, 2 is yes
    #hide the unobserved answers by setting them to 0
    answer_ids = answer_ids * observed.long()

    #unknown = 0, initial = 1, acquired = 2
    status_ids = observed.long() * 2 
    status_ids[rows, initial] = 1

    #combine the three embeddings
    evidence_ids = torch.arange(num_features, device=device)

    evidence = model["evidence"](evidence_ids)
    answers = model["answer"](answer_ids)
    status = model["status"](status_ids)

    tokens = evidence + answers + status

    #add a summary token before the findings
    cls = model["evidence"].weight[num_features]

    #reshape so it fits diaformer structure
    cls = cls.reshape(1, 1, 512)
    cls = cls.repeat(batch_size, 1, 1)

    tokens = torch.cat([cls, tokens], dim=1)

    #hide unobserved findings, but keep the summary token visible
    hidden_findings = ~observed
    cls_mask = torch.zeros(
        batch_size, 1, dtype=torch.bool, device=device
    )

    padding = torch.cat([cls_mask, hidden_findings], dim=1)

    #run the tokens through the transformer
    output = model["transformer"](
        tokens,
        src_key_padding_mask=padding
    )

    summary = output[:, 0] #takes the summary token from every example

    disease_scores = model["disease"](summary)
    question_scores = model["question"](summary)

    return disease_scores, question_scores

randomly select which evidences are reveleaed

In [11]:
def sample_observed(x, initial):
    observed = torch.zeros_like(x, dtype=torch.bool)

    for row in range(len(x)):  #for every patient
        #randomly select a number of questions to reveal, up to the budget
        count = torch.randint(budget, (1,)).item()
        questions = torch.randperm(num_features, device=device)

        #reveal the selected questions for this patient
        selected = questions[:count]
        observed[row, selected] = True

        #always reveal the initial finding
        observed[row, initial[row]] = True

    return observed

## training the transformer

In [12]:
lr=0.000010136
epochs=5
batch_size=32

In [13]:
optimizer = Adam(model.parameters(), lr=lr)

for epoch in range(epochs):
    model.train()
    tot_loss = 0.0

    for start in range(0, len(x_train), batch_size):
        end = start + batch_size

        #get the features and labels for the batch
        x = x_train[start:end].to(device)
        y = y_train[start:end].to(device)
        initial = initial_train[start:end].to(device)

        #get the observed evidences for this batch
        observed = sample_observed(x, initial)
        #get the scores, omit qns score as its not needed
        disease_scores, _ = forward(x, observed, initial)

        loss = cross_entropy(disease_scores, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tot_loss += loss.item() * len(x)

    model.eval()
    correct = 0
    
    #evaluate on validation
    with torch.no_grad():
        for start in range(0, len(x_val), batch_size):
            end = start + batch_size

            x = x_val[start:end].to(device)
            y = y_val[start:end].to(device)
            initial = initial_val[start:end].to(device)

            observed = torch.zeros_like(x, dtype=torch.bool)
            rows = torch.arange(len(x), device=device)
            observed[rows, initial] = True

            disease_scores, _ = forward(x, observed, initial)
            prediction = disease_scores.argmax(dim=1)

            correct += (prediction == y).sum().item()

    average_loss = tot_loss / len(x_train)
    accuracy = correct / len(x_val)

    print("Epoch:", epoch + 1)
    print("Training loss:", round(average_loss, 4))
    print("Initial-only validation accuracy:", round(accuracy, 3))

Epoch: 1
Training loss: 3.4515
Initial-only validation accuracy: 0.338
Epoch: 2
Training loss: 2.3985
Initial-only validation accuracy: 0.331
Epoch: 3
Training loss: 2.1172
Initial-only validation accuracy: 0.343
Epoch: 4
Training loss: 2.0271
Initial-only validation accuracy: 0.359
Epoch: 5
Training loss: 1.9743
Initial-only validation accuracy: 0.361


## GDFS

In [16]:
epochs2=5
candidate_qns=4

In [17]:
#freeze everything except the question head
for parameter in model.parameters():
    parameter.requires_grad = False

for parameter in model["question"].parameters():
    parameter.requires_grad = True

#allow optimization of the question head only
optimizer = Adam(model["question"].parameters(), lr=lr)

for epoch in range(epochs2):
    model.eval()
    model["question"].train()
    tot_loss = 0.0

    for start in range(0, len(x_train), batch_size):
        end = start + batch_size

        x = x_train[start:end].to(device)
        y = y_train[start:end].to(device)
        initial = initial_train[start:end].to(device)

        observed = sample_observed(x, initial)

        #sample potential questions from unobserved evidences
        available = (observed == False).float()
        candidates = torch.multinomial(available, candidate_qns, replacement=False)

        with torch.no_grad():
            #disease loss before asking
            before, _ = forward(x, observed, initial)
            before_loss = cross_entropy(before, y, reduction="none")

            #make one patient copy per candidate question
            expanded_x = x.repeat_interleave(candidate_qns, dim=0)
            expanded_y = y.repeat_interleave(candidate_qns)
            expanded_initial = initial.repeat_interleave(candidate_qns)
            expanded_observed = observed.repeat_interleave(candidate_qns, dim=0)

            #reveal one candidate answer in each copy
            rows = torch.arange(len(expanded_x), device=device)
            questions = candidates.flatten()
            expanded_observed[rows, questions] = True

            #disease loss after revealing each answer
            after, _ = forward(expanded_x, expanded_observed, expanded_initial)

            after_loss = cross_entropy(after, expanded_y, reduction="none")

            before_loss = before_loss.unsqueeze(1)
            after_loss = after_loss.reshape(len(x), candidate_qns)
            gains = before_loss - after_loss

        #teach the question head to predict those gains
        _, question_scores = forward(x, observed, initial)
        selected_scores = question_scores.gather(1, candidates)

        loss = mse_loss(selected_scores, gains)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tot_loss += loss.item() * len(x)

    average_loss = tot_loss / len(x_train)

    print("Selector epoch:", epoch + 1)
    print("Gain MSE:", round(average_loss, 5))

Selector epoch: 1
Gain MSE: 0.06373
Selector epoch: 2
Gain MSE: 0.04797
Selector epoch: 3
Gain MSE: 0.04143
Selector epoch: 4
Gain MSE: 0.04049
Selector epoch: 5
Gain MSE: 0.03569


## testing

In [18]:
file = os.path.join(data, "..", "ddxplus_raw", "release_evidences.json")

with open(file, encoding="utf-8-sig") as f:
    evidence_defs = json.load(f)

In [ ]:
@torch.no_grad()
def evaluate(all_x, all_y, all_initial, policy):
    model.eval()

    correct = np.zeros(budget + 1) #stores num of correct predictions at each step
    total = 0
    trace = []

    for start in range(0, len(all_x), batch_size):
        end = start + batch_size

        x = all_x[start:end].to(device)
        y = all_y[start:end].to(device)
        initial = all_initial[start:end].to(device)

        #only the initial finding is visible at the start
        observed = torch.zeros_like(x, dtype=torch.bool)
        rows = torch.arange(len(x), device=device)
        observed[rows, initial] = True

        for step in range(budget + 1):
            disease_scores, question_scores = forward(x, observed, initial)

            prediction = disease_scores.argmax(dim=1)
            correct[step] += (prediction == y).sum().item()
            
            #get the next question to ask for each patient, unless we are at the last step
            if step < budget:
                if policy == "random":
                    question_scores = torch.rand_like(question_scores)

                #prevent already observed questions from being selected by setting the to inf
                question_scores = question_scores.masked_fill(observed, float("-inf"))

                #select the question with the highest score
                selected = question_scores.argmax(dim=1)

                #reveal the selected answer for each patient
                observed[rows, selected] = True

                #save a details for the first patient only to check 
                if start == 0:
                    updated, _ = forward(x, observed, initial)
                    probabilities = updated[0].softmax(dim=0)
                    values, indices = probabilities.topk(5)

                    top_5 = []

                    for probability, index in zip(
                        values.tolist(), indices.tolist()
                    ):
                        top_5.append({
                            "disease": diseases[index],
                            "probability": probability
                        })

                    column = selected[0].item()
                    evidence_id = B_features[column]

                    trace.append({
                        "turn": step + 1,
                        "evidence_id": evidence_id,
                        "question": evidence_defs[evidence_id]["question_en"],
                        "answer": bool(x[0, column].item()),
                        "top_5": top_5
                    })

        total += len(x)

    accuracy = correct / total
    return accuracy, trace

In [20]:
val_greedy, _ = evaluate(
    x_val, y_val, initial_val, "greedy"
)

torch.manual_seed(42)

val_random, _ = evaluate(
    x_val, y_val, initial_val, "random"
)

pd.DataFrame({
    "follow_up_questions": [0, 5, 10],
    "greedy_accuracy": val_greedy[[0, 5, 10]],
    "random_accuracy": val_random[[0, 5, 10]]
})

,follow_up_questions,greedy_accuracy,random_accuracy
0,0,0.361,0.361
1,5,0.384,0.381
2,10,0.416,0.409
